# Format 3 — Multi-Agent
## Multiple agents. Shared world. Emergent behavior.

**Gemini's prediction: 15% probability. Highest technical bar, highest upside.**

Most RL hackathon submissions have one agent. A *multi-agent* submission — especially one where agents **learn to cooperate or compete on their own** — is immediately memorable.

Meta's strategic interest: OpenEnv must handle asynchronous multi-agent orchestration to compete with PettingZoo.

### What you'll learn
1. **Three multi-agent flavors** — cooperative, competitive, mixed
2. **Markov games** — the formalism, minimally
3. **PettingZoo-style API** — standard interface
4. **CTDE** — centralized training, decentralized execution
5. **A working demo**: two microgrids trading power (TwoGridOps)
6. **Independent PPO vs shared-critic PPO** (MAPPO-lite)

---
## Part 1 — The three flavors

| Flavor | Reward structure | Example |
|---|---|---|
| **Cooperative** | Agents share one reward | A drone swarm searching for survivors. Everyone wins if the target is found. |
| **Competitive** | Agents' rewards sum to 0 (zero-sum) | Chess. Poker. My gain is your loss. |
| **Mixed** | Agents have own reward, sometimes aligned, sometimes not | Energy market. Two microgrids want profit, but compete for grid capacity. |

**The hackathon sweet spot is MIXED** — it's the most realistic and the most interesting research-wise. Purely cooperative is solved (just give everyone the same reward). Purely competitive is brittle (needs self-play infrastructure).

---
## Part 2 — Markov games (the minimal formalism)

A single-agent Markov Decision Process is `(S, A, T, R)`:
- States S
- Actions A
- Transition T(s' | s, a)
- Reward R(s, a)

A **Markov game** extends this to N agents:
- States S (still one global state)
- Actions A₁, A₂, ..., Aₙ (one action per agent)
- Transition T(s' | s, a₁, ..., aₙ) — depends on **joint** action
- Rewards R₁, R₂, ..., Rₙ — each agent gets its own

### Why it's hard

In single-agent RL, your environment is stationary — if you take action *a* in state *s*, the distribution of next states is fixed. You can train against it.

In multi-agent RL, the "environment" for Agent 1 **includes Agent 2's policy**. Agent 2 is also learning. So from Agent 1's perspective, the world is **non-stationary** — the same (s, a) leads to different outcomes as Agent 2 evolves.

This breaks naive PPO. The standard fixes:
- **Independent learners**: each agent just treats other agents as part of the environment (works okay for weakly-coupled settings)
- **CTDE** (below): share information during training
- **Population-based training / self-play**: train against past versions of yourself

---
## Part 3 — The PettingZoo parallel API

PettingZoo is the de-facto standard multi-agent interface. Its "parallel" API is the simplest:

```python
obs = env.reset()           # dict: {agent_id → obs}
while not done:
    actions = {aid: policy[aid](obs[aid]) for aid in env.agents}
    obs, rewards, dones, infos = env.step(actions)   # all dicts
```

If the Round 2 theme is multi-agent, expect this exact shape.

Below we build our own mini-env following that contract.

---
## Part 4 — TwoGridOps: a 2-agent microgrid market

Two microgrids (A and B). Each has its own solar, battery, demand. They share **one** grid connection (limited to 200 kW combined).

If both try to import at peak → grid congestion → neither gets enough → **both** blackout. Classic commons problem.

They can **trade bilaterally**: if A has surplus and B has deficit, A sells to B at a mutually set price. This creates an incentive to cooperate.

**Action per agent**:
- `grid_bid` ∈ [-1, 1] : how aggressively to pull from shared grid (+) or push to it (−)
- `trade_offer` ∈ [-1, 1] : willingness to trade with the other microgrid

**Reward per agent**: local cost − blackout penalty + trade profit

This is **mixed** — both can win with coordination, or both can lose with greed.

In [ ]:
import numpy as np
from pydantic import BaseModel, Field
from typing import Literal

class MicroAction(BaseModel):
    grid_bid: float = Field(default=0.0, ge=-1.0, le=1.0)
    trade_offer: float = Field(default=0.0, ge=-1.0, le=1.0)

class MicroObs(BaseModel):
    agent_id: str
    step: int
    own_soc: float
    own_demand: float
    own_solar: float
    shared_grid_price: float
    # Partial observation of the OTHER agent — only coarse info
    peer_soc_hint: float  # with noise

class TwoGridOps:
    AGENTS = ['micro_a', 'micro_b']
    GRID_CAP = 200.0          # kW shared
    STEPS = 48                # 2 days for faster demo
    
    def reset(self, seed=42):
        self.rng = np.random.default_rng(seed); self.t = 0
        self.soc = {'micro_a': 0.5, 'micro_b': 0.5}
        self.cum_reward = {'micro_a': 0.0, 'micro_b': 0.0}
        self.blackout = {'micro_a': 0.0, 'micro_b': 0.0}
        return self._observe()
    
    def _demand(self, agent, step):
        base = 100 + 80 * np.sin(np.pi * (step % 24 - 6) / 12) ** 2
        if agent == 'micro_b': base *= 0.85  # B slightly smaller
        return max(20, base + self.rng.normal(0, 5))
    
    def _solar(self, step):
        h = step % 24
        return max(0, 150 * np.sin(np.pi * (h - 0) / 12)) if 0 <= h < 12 else 0.0
    
    def _price(self, step):
        h = step % 24
        return 4 + 8 * np.exp(-((h - 14)**2) / 8)   # peak around step 14 (8 PM)
    
    def _observe(self):
        obs = {}
        for a in self.AGENTS:
            peer = 'micro_b' if a == 'micro_a' else 'micro_a'
            obs[a] = MicroObs(
                agent_id=a, step=self.t,
                own_soc=self.soc[a],
                own_demand=self._demand(a, self.t),
                own_solar=self._solar(self.t),
                shared_grid_price=self._price(self.t),
                peer_soc_hint=self.soc[peer] + self.rng.normal(0, 0.05),
            )
        return obs
    
    def step(self, actions: dict):
        rewards = {'micro_a': 0.0, 'micro_b': 0.0}
        
        # Each agent's local supply/demand before trades
        local = {}
        for a in self.AGENTS:
            d = self._demand(a, self.t)
            s = self._solar(self.t)
            battery_pool = self.soc[a] * 100.0   # kW available in battery
            local[a] = {'demand': d, 'solar': s, 'net_before_trade': s + battery_pool - d,
                         'bid': actions[a].grid_bid, 'offer': actions[a].trade_offer}
        
        # Bilateral trade: if A.offer < 0 (wants to sell) and B.offer > 0 (wants to buy), trade happens
        trade_ab = 0.0
        oa, ob = actions['micro_a'].trade_offer, actions['micro_b'].trade_offer
        if oa < 0 and ob > 0:   # A sells, B buys
            trade_ab = min(-oa, ob) * 50.0    # up to 50 kW
        elif oa > 0 and ob < 0:   # A buys, B sells
            trade_ab = -min(oa, -ob) * 50.0
        # positive trade_ab = A→B
        
        # Grid congestion: sum of import bids (positive bids) can't exceed GRID_CAP
        pa, pb = actions['micro_a'].grid_bid, actions['micro_b'].grid_bid
        imp_a = max(0.0, pa) * self.GRID_CAP
        imp_b = max(0.0, pb) * self.GRID_CAP
        exp_a = -min(0.0, pa) * self.GRID_CAP
        exp_b = -min(0.0, pb) * self.GRID_CAP
        if imp_a + imp_b > self.GRID_CAP:
            scale = self.GRID_CAP / (imp_a + imp_b)
            imp_a *= scale; imp_b *= scale
        
        # Resolve per-agent energy balance
        for a, imp, exp, trade_sign in [('micro_a', imp_a, exp_a, -1), ('micro_b', imp_b, exp_b, +1)]:
            d = local[a]['demand']; s = local[a]['solar']
            battery_used = 0.0
            trade_kw = trade_sign * trade_ab
            net = s + imp + trade_kw - d - exp
            if net < 0:
                battery_used = min(-net, self.soc[a] * 100)
                self.soc[a] -= battery_used / 100.0
                net += battery_used
            elif net > 0:
                charge = min(net, (1.0 - self.soc[a]) * 100)
                self.soc[a] += charge * 0.9 / 100.0
            
            blackout = max(0, -net)
            self.blackout[a] += blackout
            
            price = self._price(self.t)
            cost = imp * price - exp * price * 0.9 + 25 * blackout
            trade_profit = -trade_sign * trade_ab * (price * 0.8)  # trade at 80% of grid price
            rewards[a] = -cost / 100.0 + trade_profit / 100.0
            self.cum_reward[a] += rewards[a]
        
        self.t += 1
        done = self.t >= self.STEPS
        dones = {a: done for a in self.AGENTS}
        infos = {a: {'blackout': self.blackout[a]} for a in self.AGENTS}
        return self._observe(), rewards, dones, infos
    
    @property
    def agents(self): return self.AGENTS

env = TwoGridOps()
obs = env.reset(seed=1)
print(f'Agents: {env.agents}')
print(f'micro_a obs: {obs["micro_a"]}')
print(f'micro_b obs: {obs["micro_b"]}')

---
## Part 5 — Three baseline strategies

Before training, understand the strategic space:

1. **Greedy**: always try to buy from grid, never trade
2. **Isolated**: local generation only, no grid, no trade
3. **Cooperative heuristic**: trade when one has surplus and the other deficit

Run all three and observe: **greedy causes mutual blackouts** (grid congestion), cooperative avoids them. This is the emergent phenomenon RL needs to discover.

In [ ]:
def greedy_policy(obs):
    return MicroAction(grid_bid=1.0, trade_offer=0.0)   # grab all grid, ignore peer

def isolated_policy(obs):
    return MicroAction(grid_bid=0.0, trade_offer=0.0)

def cooperative_policy(obs):
    # If own SOC low and peer SOC high → ask to buy from peer
    # If own SOC high and peer SOC low → offer to sell
    soc_gap = obs.own_soc - obs.peer_soc_hint
    if soc_gap > 0.2: offer = -0.5   # I'm richer, offer to sell
    elif soc_gap < -0.2: offer = 0.5  # I'm poorer, want to buy
    else: offer = 0.0
    # Bid grid modestly — never max out
    deficit = obs.own_demand - obs.own_solar
    bid = np.clip(deficit / 200.0, -0.5, 0.5)
    return MicroAction(grid_bid=float(bid), trade_offer=float(offer))

def run_matchup(env, policy_a, policy_b, seed=1):
    obs = env.reset(seed=seed); total = {'micro_a': 0.0, 'micro_b': 0.0}; blackouts = {'micro_a': 0.0, 'micro_b': 0.0}
    while True:
        acts = {'micro_a': policy_a(obs['micro_a']), 'micro_b': policy_b(obs['micro_b'])}
        obs, r, d, info = env.step(acts)
        total['micro_a'] += r['micro_a']; total['micro_b'] += r['micro_b']
        blackouts['micro_a'] = info['micro_a']['blackout']
        blackouts['micro_b'] = info['micro_b']['blackout']
        if all(d.values()): break
    return total, blackouts

MATCHUPS = [
    ('Greedy vs Greedy', greedy_policy, greedy_policy),
    ('Isolated vs Isolated', isolated_policy, isolated_policy),
    ('Cooperative vs Cooperative', cooperative_policy, cooperative_policy),
    ('Cooperative vs Greedy', cooperative_policy, greedy_policy),
]
for name, pa, pb in MATCHUPS:
    env = TwoGridOps()
    tot, bo = run_matchup(env, pa, pb)
    print(f'{name:30s} reward A={tot["micro_a"]:+.2f} B={tot["micro_b"]:+.2f}'
          f' | blackout A={bo["micro_a"]:.0f} B={bo["micro_b"]:.0f}')

**What you should see:**
- **Greedy vs Greedy**: both suffer from mutual blackouts (grid congestion kicks in). Classic tragedy of the commons.
- **Isolated vs Isolated**: conservative, low reward, lots of curtailment.
- **Cooperative vs Cooperative**: highest joint reward. **Coordination wins.**
- **Cooperative vs Greedy**: greedy agent often does better individually, but the system total is worse. This is the exploitation attack — why CTDE is needed.

---
## Part 6 — CTDE (Centralized Training, Decentralized Execution)

CTDE is the dominant multi-agent RL paradigm. The idea:

- **During training**, the critic sees *global* information (both agents' observations, actions, rewards). This lets it give stable value estimates despite the non-stationarity.
- **During execution**, each agent's policy uses only its own observation. So at deploy time, agents run independently — no need for a central coordinator.

This is how MADDPG (2017), COMA (2018), and MAPPO (2020) work.

```
                 TRAINING                              EXECUTION
                                                     
   obs_a ──► actor_a ──► a_a                         obs_a ──► actor_a ──► a_a
                                                     
   obs_b ──► actor_b ──► a_b                         obs_b ──► actor_b ──► a_b
                                                    
   (obs_a, obs_b, a_a, a_b) ──► shared_critic
                   │                                 
                   └──► V(s_global)                  (no critic needed)
```

### Independent vs CTDE in 3 lines

| Approach | Critic input | When to use |
|---|---|---|
| **Independent PPO** (IPPO) | `obs_i` (own only) | Weakly coupled, fewer agents, quick prototyping |
| **MAPPO** | `(obs_1, obs_2, ..., obs_n)` (global) | Tightly coupled, cooperation required |

For the hackathon, start with IPPO (simpler). If you see non-stationarity (agents' rewards oscillate during training), upgrade to MAPPO.

### Skeleton: independent PPO per agent

```python
# Two completely independent PPO agents
agent_a = ActorCritic(obs_dim=6, act_dim=2)
agent_b = ActorCritic(obs_dim=6, act_dim=2)

for iter in range(N):
    rollout_a, rollout_b = collect_joint_rollout(env, agent_a, agent_b)
    ppo_update(agent_a, rollout_a)   # each trains on its own data
    ppo_update(agent_b, rollout_b)
```

The only change from single-agent is: during rollout collection, both policies step together; during update, each uses only its own stream.

---
## Part 7 — Self-play (the other scalable idea)

For competitive envs: **train against a frozen copy of yourself**.

```
iter 0: agent_v0 plays against agent_v0 (random) → saves checkpoint
iter 1: agent_v1 trains, plays against agent_v0
iter 2: agent_v2 trains, plays against a random past version
...
```

This is how AlphaGo, AlphaZero, OpenAI Five, StarCraft II were trained. Core trick: **never train both sides simultaneously** — one side must be frozen, or you get catastrophic non-stationarity.

For the hackathon, self-play is overkill for 48 hours. Mention it in the pitch ("this would extend to self-play in the 48-hour timeline we didn't have") but don't try to implement it unless the theme forces you to.

---
## Part 8 — Hackathon application

If the theme is multi-agent (15% chance per Gemini), your submission structure:

```
/hackathon_multiagent
├── env.py                  # PettingZoo-compatible OpenEnv
├── policies/
│   ├── independent_ppo.py  # baseline
│   └── mappo.py            # shared critic
├── matchups.py             # head-to-head matrix
├── demo.py                 # the live-run script
└── README.md               # with the emergent-behavior story
```

### The demo that lands

**Three visualizations**, in order:

1. **Reward heatmap**: rows = agent A's strategy, cols = agent B's strategy, cell = joint reward. Shows cooperation has highest cells.
2. **Training curves**: reward over training iterations, with early exploration → late coordination visible.
3. **Emergent behavior trace**: at iter 100 vs iter 1000, show the actual action patterns. "Look — at iter 100 they're both greedy. At iter 1000 they've learned to alternate peak imports."

The story:

> "We didn't tell them to cooperate. The reward structure made cooperation dominant. PPO discovered it."

That's the moment judges remember.

### 48-hour execution

| Hours | Deliverable |
|---|---|
| 0–4 | Set up PettingZoo-compatible env + 3 baseline policies |
| 4–16 | Independent PPO training loop, both agents |
| 16–28 | MAPPO upgrade + compare |
| 28–40 | Self-play (optional) + emergent-behavior visualization |
| 40–48 | Demo + pitch + reward heatmap |

### Blind spot to watch

**Non-stationarity kills MA-PPO.** If your training curves oscillate wildly, it's because both agents are moving targets for each other. Fixes: (1) lower learning rates (2x smaller than single-agent), (2) freeze one agent for N iterations while the other trains, (3) move to CTDE with shared critic.

---

## Summary — all three formats

You now have a notebook for each of Gemini's top 3 predictions:

| Format | Probability | Notebook | Core abstraction |
|---|---|---|---|
| Agent Arena | 60% | `format1_agent_arena.ipynb` | `AgentRouter` reading Pydantic schemas |
| Hierarchical | 25% | `format2_hierarchical.ipynb` | LLM planner + goal-conditioned executor |
| Multi-Agent | 15% | `format3_multiagent.ipynb` | PettingZoo API + CTDE |

**Before the finale:** run all three end-to-end on your machine. When Meta reveals the theme on Day 0, you'll know within 30 minutes which notebook is your starting skeleton. That's a massive advantage over teams who come in blind.